# mizuRoute vs. USGS Observed Streamflow -- Monthly Comparison
Uses the `flow_multi_agency_all_meta.csv` crosswalk (gauge -> merit_reach_id) validated earlier, pulls monthly mizuRoute `RoutedRunoff`, compares against the local combined multi-agency observed streamflow dataset, and computes NSE/KGE per gauge.

**Fixed in this version (see inline notes marked `# FIX:`):**
1. `reachID` in the mizuRoute output carries a spurious `time` dimension even though it's physically constant. The original per-gauge lookup (`np.where(ds_rof["reachID"].values == reach_id)[0]`) silently indexed into the *time* axis instead of the *seg* axis, so every gauge comparison was evaluating segment 0 regardless of which reach was requested. Fixed by slicing `reachID` to a single timestep once, up front.
2. The local combined obs file (`flow_multi_agency_*.nc`, 1965-2024) was loaded but never actually used -- the notebook queried the live USGS NWIS web API every time instead, which is slower and depends on outbound network access from a compute node. Now the local file is used first, with the live API as an optional fallback for sites it doesn't cover.
3. mizuRoute's monthly output is timestamped mid-month (e.g. day 16) while the observed monthly file uses a different day-of-month convention (commonly month-start). Merging on exact timestamps silently dropped nearly every month even when both series covered the same period. Both series are now normalized to a calendar-month `Period` before merging, and the "not enough overlap" failure branch now prints each series' actual length/date range so a genuine data gap is distinguishable from an alignment bug.
4. The local *monthly* obs file (`flow_multi_agency_..._month.nc`) only covers 29 of the 2311 crosswalk sites -- not usable as the primary observed source. The *daily* file covers all 2311 (an exact match to the crosswalk), so that's now the primary source, resampled to monthly here instead of relying on the mostly-empty monthly file. The live USGS API remains a fallback for anything not in the daily file.
5. Added Section 9: search the crosswalk by river name (e.g. Missouri, Columbia, Colorado) -- plus a HUC02-based lookup, since name matching alone pulled in the wrong river for "Colorado" (Texas's Colorado River, not the western Colorado) and missed the real Columbia River gauges entirely.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.dates as mdates
import geopandas as gpd
import glob
import sys
sys.path.insert(0, "../config")  # analysis/config/paths.py (run from this folder)
import paths

In [ ]:
# Exploratory / not wired into the eval pipeline below -- kept for reference.
gdf_catch = gpd.read_file(paths.MERIT_CATCH_GPKG)
print(gdf_catch.columns.tolist())
print(gdf_catch.head())
print(f"Number of catchments: {len(gdf_catch)}")

In [ ]:
CROSSWALK_CSV = paths.CROSSWALK
MIZUROUTE_DIR = paths.rof_hist("control") + "/"
MIZUROUTE_PATTERN = "control.mizuroute.h.*.nc"

# FIX: these were previously duplicated (opened again inside a later cell
# and again inside get_usgs_monthly). Defined once here.
OBS_DAILY_PATH = paths.OBS_DAILY
OBS_MONTHLY_PATH = paths.OBS_MONTHLY

OUTPUT_DIR = "streamflow_eval_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
ds_obs_daily = xr.open_dataset(OBS_DAILY_PATH, decode_times=True)
ds_obs_monthly = xr.open_dataset(OBS_MONTHLY_PATH, decode_times=True)

obs_daily_sites = set(ds_obs_daily["site"].values.astype(int))
obs_monthly_sites = set(ds_obs_monthly["site"].values.astype(int))
print(f"Local obs file covers {len(obs_monthly_sites)} sites (monthly), {len(obs_daily_sites)} sites (daily)")

In [ ]:
ds_obs_daily

In [ ]:
# FIX (diagnostic, new): confirm the actual flow variable name and units in
# the local obs file before trusting values pulled from it. The original
# notebook never inspected this -- it's easy to silently mix up units
# (cfs vs m3/s) or grab the wrong variable if there's more than one.
print("Data variables in ds_obs_daily:", list(ds_obs_daily.data_vars))
for v in ds_obs_daily.data_vars:
    print(f"  {v}: dims={ds_obs_daily[v].dims}, units={ds_obs_daily[v].attrs.get('units', '???')}")
print()
print("Data variables in ds_obs_monthly:", list(ds_obs_monthly.data_vars))
for v in ds_obs_monthly.data_vars:
    print(f"  {v}: dims={ds_obs_monthly[v].dims}, units={ds_obs_monthly[v].attrs.get('units', '???')}")

def _guess_flow_varname(ds):
    candidates = [v for v in ds.data_vars if v.lower() in
                  ("flow", "q", "discharge", "streamflow", "runoff")]
    if candidates:
        return candidates[0]
    if len(ds.data_vars) == 1:
        return list(ds.data_vars)[0]
    raise ValueError(
        f"Could not auto-detect the flow variable in ds_obs_monthly; "
        f"found {list(ds.data_vars)}. Set OBS_FLOW_VAR manually below."
    )

# FIX: the monthly file only covers 29 of 2311 crosswalk sites (confirmed
# via obs_monthly_sites above) -- the daily file, which covers all 2311, is
# the actual primary observed source (resampled to monthly in
# get_usgs_monthly below). OBS_FLOW_VAR is detected once here and used for
# both, since both files share the same variable name.
OBS_FLOW_VAR = _guess_flow_varname(ds_obs_daily)
print(f"\nUsing '{OBS_FLOW_VAR}' as the flow variable.")
print(f"Units: {ds_obs_daily[OBS_FLOW_VAR].attrs.get('units', 'NOT SET -- confirm manually before trusting comparisons')}")

In [ ]:
region='18'
shp_file   = paths.HUC2_SITES.format(region=region)
gdf = gpd.read_file(shp_file)
df_map=gdf.copy()
df_map.info()

df_map["gage_type"] = "interior"   #These are mid-network gauges. depend on upstream choices, must be evaluated after all their contributors are fixed

df_map.loc[
    (df_map["site_rank"] == 0) & (df_map["down_site_ID"].isna()),
    "gage_type"
] = "single_basin"    #Just choose best ensemble and fix HUC12s -----> No routing dependency.

df_map.loc[
    (df_map["site_rank"] == 0) & (~df_map["down_site_ID"].isna()),
    "gage_type"
] = "headwater" # have no upstream contributors, They start the sequential chain for each basin.
# (rest of gage_type classification logic as before -- unused by the eval
# pipeline below, kept for reference / other analysis)

## 2. Load gauge crosswalk

In [ ]:
df_gauges = pd.read_csv(CROSSWALK_CSV)
df_gauges["merit_reach_id"] = df_gauges["merit_reach_id"].astype(int)
df_gauges["Latitude"] = df_gauges["Latitude"].astype(float)
df_gauges["Longitude"] = df_gauges["Longitude"].astype(float)
print(f"Loaded {len(df_gauges)} gauges")
df_gauges[["Station ID", "GAGE_ID", "Long_Name", "merit_reach_id", "Latitude", "Longitude"]].head()

## 3. Load mizuRoute monthly output (all years)

In [ ]:
files_rof = sorted(glob.glob(MIZUROUTE_DIR + MIZUROUTE_PATTERN))
print(f"Found {len(files_rof)} monthly mizuRoute files")
print(files_rof[0], "...", files_rof[-1])

def _keep_rof_fields(ds):
    keep = [v for v in ["RoutedRunoff", "reachID"] if v in ds.variables]
    return ds[keep]

ds_rof = xr.open_mfdataset(
    files_rof,
    combine="nested",
    concat_dim="time",
    decode_times=True,
    parallel=True,
    preprocess=_keep_rof_fields,
)

# FIX (critical): reachID has dims ('time','seg') even though it's constant
# across time. Slice to a single timestep ONCE here, rather than repeatedly
# comparing the full 2D array inside a per-gauge lookup function -- doing
# that made np.where return the wrong axis's indices (see helper functions
# section below for what this was silently breaking).
reach_id_1d = ds_rof["reachID"].isel(time=0).compute().values.astype(int)
print(f"reach_id_1d: {len(reach_id_1d)} reaches (should match n_reaches in ds_rof)")

ds_rof

## 4. Helper functions: extract modeled series, pull observed, compute metrics

`get_modeled_monthly` now indexes against the pre-sliced `reach_id_1d` (fixed).
`get_usgs_monthly` now reads the local combined obs file first (fixed to
actually use `ds_obs_monthly`/`ds_obs_daily`, loaded once in Section 1), and
only falls back to a live USGS web query if a site isn't in the local file.

In [ ]:
def get_modeled_monthly(reach_id):
    """Extract modeled monthly RoutedRunoff (m3/s) time series for a given reachID."""
    # FIX: was np.where(ds_rof["reachID"].values == reach_id)[0], which
    # indexed the wrong (time) axis because reachID is 2D (time, seg).
    # reach_id_1d is the correct, pre-sliced 1D (seg,) array.
    idx = np.where(reach_id_1d == reach_id)[0]
    if len(idx) == 0:
        return None
    series = ds_rof["RoutedRunoff"].isel(seg=idx[0]).to_series()
    # FIX: mizuRoute monthly output is timestamped mid-month (e.g. day 16),
    # while the observed monthly file uses a different day-of-month
    # convention (commonly month-start). Concatenating on exact timestamps
    # then silently drops nearly every month even when both series cover
    # the same period. Normalize to a calendar-month Period so alignment
    # doesn't depend on which day of the month either source stamps.
    series.index = pd.PeriodIndex(pd.to_datetime(series.index.astype(str)), freq="M")
    series.name = "modeled"
    return series


def get_usgs_monthly(gage_id, start_date, end_date):
    """Observed monthly streamflow (m3/s) for a gauge. Uses the local
    combined daily file first (covers all 2311 crosswalk sites -- the
    monthly variant of this file only covers 29 of them, so it's not a
    usable primary source), resampled to monthly; falls back to a live
    USGS NWIS query only if the site isn't in the local file at all."""
    site_id = int(gage_id)

    if site_id in obs_daily_sites:
        da = ds_obs_daily[OBS_FLOW_VAR].sel(site=site_id)
        daily = da.to_series().loc[start_date:end_date].dropna()
        if not daily.empty:
            series = daily.resample("MS").mean()
            series.name = "observed"
            # FIX: same month-alignment normalization as get_modeled_monthly.
            series.index = pd.PeriodIndex(series.index, freq="M")
            return series
        # fall through to live query if the local slice is empty for this period

    return _get_usgs_monthly_live(gage_id, start_date, end_date)


def _get_usgs_monthly_live(gage_id, start_date, end_date):
    """Fallback: live USGS NWIS query, only used if the local combined
    obs file doesn't cover this gauge/period."""
    site_no = str(gage_id).zfill(8)
    url = (f"https://waterservices.usgs.gov/nwis/dv/?format=json"
           f"&sites={site_no}&startDT={start_date}&endDT={end_date}"
           f"&parameterCd=00060&siteStatus=all")
    try:
        df = pd.read_json(url)
        ts = df["value"]["timeSeries"]
        if len(ts) == 0:
            return None
        vals = ts[0]["values"][0]["value"]
        obs = pd.DataFrame(vals)
        obs["dateTime"] = pd.to_datetime(obs["dateTime"]).dt.tz_localize(None)
        obs["value"] = pd.to_numeric(obs["value"], errors="coerce") * 0.0283168  # cfs -> m3/s
        obs = obs.set_index("dateTime")["value"]
        obs_monthly = obs.resample("MS").mean()
        obs_monthly.name = "observed"
        # FIX: same month-alignment normalization.
        obs_monthly.index = pd.PeriodIndex(obs_monthly.index, freq="M")
        return obs_monthly
    except Exception as e:
        print(f"  USGS live fetch failed for site {site_no}: {e}")
        return None


def nse(obs, sim):
    return 1 - ((obs - sim) ** 2).sum() / ((obs - obs.mean()) ** 2).sum()


def kge(obs, sim):
    r = obs.corr(sim)
    alpha = sim.std() / obs.std()
    beta = sim.mean() / obs.mean()
    return 1 - ((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2) ** 0.5

## 5. Select gauges to evaluate
Edit `SELECTED_STATIONS` below -- use `Station ID` values from df_gauges. Start with a small set; expand once the pipeline is confirmed working.

In [ ]:
# SELECTED_STATIONS = ["01013500"]  # add more Station IDs as strings, e.g. ["01013500", "01030500"]

SELECTED_STATIONS = [
    "01013500",  # Fish River, ME (already tested)
    "06909000",  # Missouri R at Boonville, MO
    "06934500",  # Missouri R at Hermann, MO
    "06893000",  # Missouri R at Kansas City, MO
    "06813500",  # Missouri R at Rulo, NE
    "06090800",  # Missouri R at Fort Benton, MT
    "05331000",  # Mississippi R at St. Paul, MN
    "05378500",  # Mississippi R at Winona, MN
    "03611500",  # Ohio R at Metropolis, IL
    "03086000",  # Ohio R at Sewickley, PA
]

MODEL_START = "1980-10-01"
MODEL_END = "2018-09-30"

## 6. Run comparison for each selected gauge

In [ ]:
results_summary = []

for station_id in SELECTED_STATIONS:
    row = df_gauges[df_gauges["Station ID"] == station_id]
    if len(row) == 0:
        print(f"Station {station_id} not found in crosswalk, skipping.")
        continue
    row = row.iloc[0]
    reach_id = int(row["merit_reach_id"])
    gage_id = row["GAGE_ID"]
    long_name = row["Long_Name"]

    print(f"\n--- {station_id} ({long_name}) -- reach {reach_id} ---")

    modeled = get_modeled_monthly(reach_id)
    if modeled is None:
        print("  No modeled data found for this reachID.")
        continue

    observed = get_usgs_monthly(gage_id, MODEL_START, MODEL_END)
    if observed is None or observed.dropna().empty:
        print("  No observed data available.")
        continue

    combined = pd.concat([observed, modeled], axis=1).dropna()
    if len(combined) < 12:
        # FIX (diagnostic, new): show why, instead of just failing silently.
        # A real data-gap looks like non-overlapping date ranges below; an
        # alignment bug (already fixed above) would have shown both series
        # covering the same years but still failing to merge.
        print(f"  Not enough overlapping months to evaluate "
              f"({len(combined)} months after merge).")
        print(f"    modeled:  n={len(modeled)}, range {modeled.index.min()} to {modeled.index.max()}")
        print(f"    observed: n={len(observed)}, range {observed.index.min()} to {observed.index.max()}")
        continue

    nse_val = nse(combined["observed"], combined["modeled"])
    kge_val = kge(combined["observed"], combined["modeled"])
    bias_pct = 100 * (combined["modeled"].mean() - combined["observed"].mean()) / combined["observed"].mean()

    print(f"  NSE={nse_val:.3f}  KGE={kge_val:.3f}  Bias={bias_pct:.1f}%  n_months={len(combined)}")

    results_summary.append({
        "Station ID": station_id,
        "Long_Name": long_name,
        "merit_reach_id": reach_id,
        "GAGE_ID": gage_id,
        "NSE": nse_val,
        "KGE": kge_val,
        "Bias_pct": bias_pct,
        "n_months": len(combined),
    })

    fig, ax = plt.subplots(figsize=(14, 4))
    combined["observed"].plot(ax=ax, label="Observed", color="black", linewidth=1.2)
    combined["modeled"].plot(ax=ax, label="Modeled (mizuRoute)", color="tab:blue", linewidth=1.2, alpha=0.8)
    ax.set_ylabel("Streamflow (m3/s)")
    ax.set_title(f"{station_id} -- {long_name}\nNSE={nse_val:.2f}, KGE={kge_val:.2f}, Bias={bias_pct:.1f}%")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/{station_id}_monthly_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Summary table across all evaluated gauges

In [ ]:
df_summary = pd.DataFrame(results_summary)
df_summary = df_summary.sort_values("NSE", ascending=False)
df_summary.to_csv(f"{OUTPUT_DIR}/streamflow_eval_summary.csv", index=False)
print(f"Saved summary for {len(df_summary)} gauges to {OUTPUT_DIR}/streamflow_eval_summary.csv")
df_summary

In [ ]:
missouri_stations = ["06909000", "06934500", "06893000", "06813500", "06090800"]

df_gauges[df_gauges["Station ID"].isin(missouri_stations)][
    ["Station ID", "Long_Name", "Type (naturalized, observed)", "source"]
]

## 8. Scaling up
To evaluate more gauges, add more Station IDs to `SELECTED_STATIONS` in section 5 and re-run sections 6-7. For a first broad pass, consider sampling a handful of gauges across different HUC02 regions (`df_gauges['HUC02'].unique()`) to cover diverse hydroclimate zones, rather than all 2311 at once.

## 9. Find specific named rivers (new)
Search the crosswalk by name instead of guessing which `reachID` corresponds to a given river. Useful for pulling a specific major basin (Missouri, Columbia, Colorado, etc.) for a hydrograph sanity check or eval, without relying on flow magnitude alone -- large rivers get split into many segments/distributaries near their mouths, so a plain top-N-by-flow search tends to resurface the same river repeatedly rather than finding distinct basins.

In [ ]:
RIVER_NAMES_OF_INTEREST = ["Missouri", "Columbia", "Colorado", "Mississippi", "Ohio"]

for name in RIVER_NAMES_OF_INTEREST:
    matches = df_gauges[df_gauges["Long_Name"].str.contains(name, case=False, na=False)]
    print(f"\n{name}: {len(matches)} matching gauge(s)")
    if len(matches):
        display(matches[["Station ID", "GAGE_ID", "Long_Name", "merit_reach_id",
                          "Latitude", "Longitude"]].head(10))

### 9b. Name matching alone isn't reliable -- filter by HUC02 too
Plain substring search on `Long_Name` pulled in the wrong river: all 23 "Colorado" matches above are the Colorado River in **Texas** (HUC02 12), not the western Colorado River through the Grand Canyon (HUC02 14/15) -- and "Columbia" matched a town name, missing the real Columbia River (HUC02 17) mainstem gauges entirely. Combine name with HUC02 to disambiguate.

Reference HUC02 regions: 10 = Missouri, 14/15 = Upper/Lower Colorado, 17 = Pacific Northwest (Columbia), 05/06/08 = Ohio/Tennessee/Lower Mississippi, 07 = Upper Mississippi.

In [ ]:
if "HUC02" in df_gauges.columns:
    TARGET_BASINS = {
        "Missouri":    {"huc02": ["10"]},
        "Colorado":    {"huc02": ["14", "15"]},
        "Columbia":    {"huc02": ["17"]},
        "Mississippi": {"huc02": ["05", "06", "07", "08"]},
    }
    for name, spec in TARGET_BASINS.items():
        matches = df_gauges[
            df_gauges["Long_Name"].str.contains(name, case=False, na=False)
            & df_gauges["HUC02"].astype(str).str.zfill(2).isin(spec["huc02"])
        ]
        print(f"\n{name} (HUC02 in {spec['huc02']}): {len(matches)} matching gauge(s)")
        if len(matches):
            display(matches[["Station ID", "GAGE_ID", "Long_Name", "merit_reach_id",
                              "HUC02", "Latitude", "Longitude"]].head(10))
else:
    print("df_gauges has no 'HUC02' column -- check the actual column name "
          "(df_gauges.columns.tolist()) and adjust above.")